<a href="https://colab.research.google.com/github/antonypradeep54/Agent-LangGraph-Competitive-Footfall-Insights/blob/main/C13_M4_L9_Project_LangGraph_Competitive_Footfall_Insights.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<font size=+4 color="#009688"><center><b> PROJECT </b></center></font>

<font size=+3 color="#009688"><center><b> LangGraph - Competitive Footfall Insights</b></center></font>

## Table of contents

- Problem statement
- Libraries and configurations
- Data prepration
- Defining agents
    - Query agent
    - Analytics agent
    - Report agent
    - Output agent
    - Critic agent
    - User proxy agent
- Configuring agent connectivity
- Agent orchestration and pipeline execution
- References

##Problem Statement

Businesses in competitive locations often need to monitor their market rivals closely to stay relevant
and maximize customer engagement. Specifically, clothing stores in bustling areas face intense
competition, making it essential to understand the local landscape of competitors, their footfall
trends, and peak hours. </br> </br>
This code is designed to address this challenge by creating a conversational AI pipeline that allows
users to query and generate a report on nearby clothing store competitors. The system integrates a
language model with a search tool to provide real-time data on competitors, including store footfall
and busiest times, giving actionable insights to enhance business strategies.

</br>
This solution should benefit the following groups:</br></br>
1. Business Owners and Managers </br>
Clothing store owners or managers in areas with high competition can use this report to
identify key competitors, analyze peak customer hours, and adjust their own strategies to
improve market position. </br></br>
2. Marketing and Strategy Teams </br>
Marketing teams responsible for local engagement can leverage insights on footfall trends to
optimize promotions during high-traffic periods. Additionally, competitive data can help
strategists shape targeted campaigns, aligning with or counteracting competitor timings. </br></br>
3. Real Estate and Location Analysts </br>
Location-based data on competitor footfall and busy hours can also assist analysts in
recommending ideal store locations for future openings, as well as identifying high-value
times for new store launches. </br></br>
4. Investors and Market Analysts </br>
Investors considering opportunities in retail can use competitor analysis to assess market
saturation, business potential, and potential risks in high-traffic areas like Koramangala,
Bangalore. </br>

##Libraries and configurations

In [ ]:
%%capture
!pip install -U langgraph langchain langchain-core langchain-openai
!pip install langchain-community tavily-python
!pip install osmnx --quiet
!pip install --upgrade osmnx

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import ToolNode
from langchain_community.tools import TavilySearchResults
from IPython.display import Image

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OpenAI_API_Key')

In [ ]:
import osmnx as ox
import pandas as pd

In [ ]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

##Data preparation

In [ ]:
import osmnx as ox
import geopandas as gpd
from shapely.geometry import Point

# Define address and search radius
address = "Burj Khalifa, Dubai, United Arab Emirates"
dist = 1000  # meters

# Define tags
shop_tags = {"shop": "clothes"}
amenity_tags = {"amenity": True}

# Fetch data
stores = ox.features_from_address(address, dist=dist, tags=shop_tags)
amenities = ox.features_from_address(address, dist=dist, tags=amenity_tags)

# Clean & prepare
stores = stores[["name", "geometry"]].dropna(subset=["name"]).reset_index(drop=True)
stores["lat"] = stores.geometry.centroid.y
stores["lon"] = stores.geometry.centroid.x
amenities = amenities[["geometry"]].reset_index(drop=True)

# Project both layers to a metric CRS (UTM)
# Use UTM Zone for Dubai (EPSG:32640 or auto detect)
stores = stores.to_crs(epsg=32640)
amenities = amenities.to_crs(epsg=32640)

# Compute amenities within 100m buffer for each store
footfall_scores = []
for idx, store in stores.iterrows():
    buffer = store.geometry.buffer(100)  # 100 meters now truly 100m
    nearby_count = amenities[amenities.intersects(buffer)].shape[0]
    footfall_scores.append(nearby_count)

stores["footfall_score"] = footfall_scores

# Sort by estimated footfall
stores = stores.sort_values("footfall_score", ascending=False)

print(stores[["name", "footfall_score"]].head())

In [ ]:
stores.head(5)

##Query Agent

In [ ]:
import re
from geopy.distance import geodesic

In [ ]:
class QueryAgent:
    def __init__(self, dataframe):
        self.df = dataframe

    def parse_query(self, query):
        """
        Extracts parameters from the query:
        - top N stores
        - location filter (lat/lon and radius)
        """
        top_n = 50
        lat = None
        lon = None
        radius_km = None

        # Extract top N
        top_match = re.search(r"top (\d+)", query, re.IGNORECASE)
        if top_match:
            top_n = int(top_match.group(1))

        # Extract lat/lon and radius
        coord_match = re.search(r"lat\s*([0-9.]+)\s*,\s*lon\s*([0-9.]+)", query, re.IGNORECASE)
        if coord_match:
            lat = float(coord_match.group(1))
            lon = float(coord_match.group(2))
        radius_match = re.search(r"within ([0-9.]+)\s*km", query, re.IGNORECASE)
        if radius_match:
            radius_km = float(radius_match.group(1))

        return {"top_n": top_n, "lat": lat, "lon": lon, "radius_km": radius_km}

    def get_stores(self, query):
        params = self.parse_query(query)
        df_filtered = self.df.copy()

        # Apply location radius filter
        if params["lat"] is not None and params["lon"] is not None and params["radius_km"] is not None:
            def is_within(row):
                return geodesic((params["lat"], params["lon"]), (row["lat"], row["lon"])).km <= params["radius_km"]
            df_filtered = df_filtered[df_filtered.apply(is_within, axis=1)]

        # Sort by footfall_score and pick top N
        result = df_filtered.sort_values(by="footfall_score", ascending=False).head(params["top_n"])
        return result[["name", "lat", "lon", "footfall_score"]]

# -----------------------------
# Test Query Agent
# -----------------------------
query_agent = QueryAgent(stores)

query1 = "Show me top 3 stores"
query2 = "Show me top 2 stores near lat 25.198, lon 55.279 within 0.1 km"

print("Query 1 result:")
print(query_agent.get_stores(query1))
print("\nQuery 2 result:")
print(query_agent.get_stores(query2))

##Analytics Agent

In [ ]:
class AnalyticsAgent:
    def __init__(self, stores_df):
        """
        stores_df: pandas DataFrame containing store data with columns:
        ['name', 'lat', 'lon', 'footfall_score']
        """
        self.stores_df = stores_df.copy()

    def top_stores(self, n=5):
        """Return top N stores by footfall_score"""
        return self.stores_df.sort_values(by="footfall_score", ascending=False).head(n)

    def stores_within_radius(self, lat, lon, radius_km):
        """Return stores within a radius (km) from a given point"""
        def is_within(row):
            return geodesic((lat, lon), (row["lat"], row["lon"])).km <= radius_km

        return self.stores_df[self.stores_df.apply(is_within, axis=1)]

    def average_footfall(self):
        """Return the average footfall across all stores"""
        return self.stores_df["footfall_score"].mean()

    def summary(self):
        """Return a summary of the dataset"""
        return {
            "total_stores": len(self.stores_df),
            "average_footfall": self.average_footfall(),
            "top_store": self.top_stores(1).iloc[0].to_dict()
        }

    def query_analytics(self, query):
        """
        Example queries:
        - "Top 3 stores by footfall"
        - "Stores within 1 km of 25.198, 55.279"
        - "Average footfall"
        """
        query = query.lower()

        if "top" in query:
            import re
            match = re.search(r"top (\d+)", query)
            n = int(match.group(1)) if match else 5
            return self.top_stores(n)
        elif "within" in query and "km" in query:
            import re
            match = re.search(r"within ([\d.]+) km of ([\d.-]+),\s*([\d.-]+)", query)
            if match:
                radius = float(match.group(1))
                lat = float(match.group(2))
                lon = float(match.group(3))
                return self.stores_within_radius(lat, lon, radius)
        elif "average" in query:
            return self.average_footfall()
        else:
            return self.summary()

In [ ]:
# Initialize the agent
analytics_agent = AnalyticsAgent(stores)

# Top 3 stores
print(analytics_agent.query_analytics("Top 3 stores by footfall"))

# Stores within 1 km of a location
print(analytics_agent.query_analytics("Stores within 1 km of 25.198, 55.279"))

# Average footfall
print(analytics_agent.query_analytics("Average footfall"))

# Full summary
print(analytics_agent.query_analytics("Summary"))

##Report agent

In [ ]:
class ReportAgent:
    def __init__(self, analytics_agent):
        """
        analytics_agent: an instance of AnalyticsAgent
        """
        self.analytics_agent = analytics_agent

    def generate_text_report(self):
        """Generate a textual summary report"""
        summary = self.analytics_agent.summary()
        report = f"""
        === Store Analytics Report ===
        Total Stores: {summary['total_stores']}
        Average Footfall: {summary['average_footfall']:.2f}
        Top Store: {summary['top_store']['name']} with footfall {summary['top_store']['footfall_score']}
        """
        return report.strip()

    def generate_table_report(self, top_n=5):
        """Generate a table report of top N stores"""
        top_stores = self.analytics_agent.top_stores(top_n)
        return top_stores

    def generate_bar_chart(self, top_n=5):
        """Generate a bar chart of top N stores by footfall"""
        top_stores = self.analytics_agent.top_stores(top_n)
        plt.figure(figsize=(8,5))
        sns.barplot(x='footfall_score', y='name', data=top_stores, palette="viridis")
        plt.title(f"Top {top_n} Stores by Footfall")
        plt.xlabel("Footfall Score")
        plt.ylabel("Store Name")
        plt.tight_layout()
        plt.show()

    def generate_full_report(self, top_n=5, show_chart=True):
        """Generate full report: text + table + optional chart"""
        print(self.generate_text_report())
        print("\n--- Top Stores Table ---")
        print(self.generate_table_report(top_n))
        if show_chart:
            self.generate_bar_chart(top_n)

##Output agent

In [ ]:
class OutputAgent:
    def __init__(self, report_agent):
        """
        report_agent: an instance of ReportAgent
        """
        self.report_agent = report_agent

    def output_console(self, top_n=5, show_chart=True):
        """Print the full report to console"""
        self.report_agent.generate_full_report(top_n=top_n, show_chart=show_chart)

    def output_csv(self, filepath="report.csv", top_n=5):
        """Save top stores table as CSV"""
        table = self.report_agent.generate_table_report(top_n=top_n)
        table.to_csv(filepath, index=False)
        print(f"Report saved as CSV at {filepath}")

    def output_excel(self, filepath="report.xlsx", top_n=5):
        """Save top stores table as Excel"""
        table = self.report_agent.generate_table_report(top_n=top_n)
        table.to_excel(filepath, index=False)
        print(f"Report saved as Excel at {filepath}")

    def output_dict(self, top_n=5):
        """Return report as a Python dictionary"""
        summary = self.report_agent.analytics_agent.summary()
        top_stores = self.report_agent.generate_table_report(top_n=top_n).to_dict(orient="records")
        return {
            "summary": summary,
            "top_stores": top_stores
        }

##Critics agent

In [ ]:
class CriticAgent:
    def __init__(self, analytics_agent):
        """
        analytics_agent: instance of AnalyticsAgent to validate its outputs
        """
        self.analytics_agent = analytics_agent

    def validate_data(self):
        """
        Validate store data for anomalies.
        Checks for:
        - Missing lat/lon or footfall
        - Negative footfall values
        - Extremely high or low footfall
        Returns a list of warnings.
        """
        df = self.analytics_agent.stores_df
        warnings = []

        if df.empty:
            warnings.append("Warning: No store data available.")

        if df[['lat', 'lon', 'footfall_score']].isnull().any().any():
            warnings.append("Warning: Some stores have missing latitude, longitude, or footfall.")

        if (df['footfall_score'] < 0).any():
            warnings.append("Warning: Some stores have negative footfall scores.")

        # Optional: check for extreme outliers
        mean_footfall = df['footfall_score'].mean()
        std_footfall = df['footfall_score'].std()
        outliers = df[(df['footfall_score'] > mean_footfall + 3*std_footfall) |
                      (df['footfall_score'] < mean_footfall - 3*std_footfall)]
        if not outliers.empty:
            warnings.append(f"Warning: Found {len(outliers)} stores with unusually high or low footfall.")

        return warnings

    def provide_feedback(self, report_agent):
        """
        Optionally provide feedback to the ReportAgent.
        Can append warnings or modify the report.
        """
        warnings = self.validate_data()
        if warnings:
            print("\n=== CriticAgent Feedback ===")
            for w in warnings:
                print(w)
        else:
            print("CriticAgent: All data looks fine.")

##User proxy agent

In [ ]:
class UserProxyAgent:
    def __init__(self, query_agent, analytics_agent, report_agent, output_agent, critic_agent):
        self.query_agent = query_agent
        self.analytics_agent = analytics_agent
        self.report_agent = report_agent
        self.output_agent = output_agent
        self.critic_agent = critic_agent

    def handle_query(self, user_query, top_n=5, output_format="dict"):
        """
        End-to-end query processing with conversation logging
        Returns a structured conversation log.
        """
        conversation = []

        # 1️⃣ User message
        conversation.append({"agent": "User", "message": user_query})

        # 2️⃣ QueryAgent fetches stores
        conversation.append({"agent": "QueryAgent", "message": f"Fetching stores for query: '{user_query}'"})
        stores_df = self.query_agent.get_stores(user_query)
        conversation.append({"agent": "QueryAgent", "message": f"Found {len(stores_df)} stores"})

        # 3️⃣ Update AnalyticsAgent
        self.analytics_agent.stores_df = stores_df
        conversation.append({"agent": "AnalyticsAgent", "message": f"Analyzing {len(stores_df)} stores"})

        # 4️⃣ CriticAgent validates data
        warnings = self.critic_agent.validate_data()
        if warnings:
            conversation.append({"agent": "CriticAgent", "message": f"Warnings: {warnings}"})
        else:
            conversation.append({"agent": "CriticAgent", "message": "Data validation passed"})

        # 5️⃣ ReportAgent generates report
        conversation.append({"agent": "ReportAgent", "message": "Generating report"})
        report_summary = self.report_agent.generate_text_report()
        top_stores_table = self.report_agent.generate_table_report(top_n)

        # 6️⃣ OutputAgent delivers report
        conversation.append({"agent": "OutputAgent", "message": "Delivering report"})
        if output_format == "console":
            print(report_summary)
            print("\n--- Top Stores Table ---")
            print(top_stores_table)
        elif output_format == "dict":
            return {
                "conversation": conversation,
                "report_summary": report_summary,
                "top_stores": top_stores_table.to_dict(orient="records")
            }
        return conversation

##Configuring agent connectivity

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# 1️⃣ Define agents as nodes
agents = [
    "UserProxyAgent",
    "QueryAgent",
    "AnalyticsAgent",
    "ReportAgent",
    "OutputAgent",
    "CriticAgent"  # Optional
]

# 2️⃣ Define edges (data flow between agents)
edges = [
    ("UserProxyAgent", "QueryAgent"),
    ("QueryAgent", "AnalyticsAgent"),
    ("AnalyticsAgent", "ReportAgent"),
    ("ReportAgent", "OutputAgent"),
    ("AnalyticsAgent", "CriticAgent"),  # Optional feedback
    ("CriticAgent", "ReportAgent")      # Optional feedback
]

# 3️⃣ Create the directed graph
G = nx.DiGraph()
G.add_nodes_from(agents)
G.add_edges_from(edges)

# 4️⃣ Draw the graph
plt.figure(figsize=(10,6))
pos = nx.spring_layout(G, seed=42)  # positions for all nodes
nx.draw_networkx_nodes(G, pos, node_size=2500, node_color='skyblue')
nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=20, edge_color='gray')
nx.draw_networkx_labels(G, pos, font_size=12, font_weight='bold')
plt.title("LangGraph Multi-Agent System")
plt.axis('off')
plt.show()

In [ ]:
# 1️⃣ Make sure your stores DataFrame exists
# stores = your DataFrame with columns ['name', 'lat', 'lon', 'footfall_score']

# 2️⃣ Initialize agents
analytics_agent = AnalyticsAgent(stores)
report_agent = ReportAgent(analytics_agent)
output_agent = OutputAgent(report_agent)
critic_agent = CriticAgent(analytics_agent)  # <-- initialize CriticAgent

# 3️⃣ Initialize UserProxyAgent with critic_agent
user_proxy = UserProxyAgent(
    query_agent=query_agent,
    analytics_agent=analytics_agent,
    report_agent=report_agent,
    output_agent=output_agent,
    critic_agent=critic_agent  # <-- pass the CriticAgent
)

# 4️⃣ Define the invoke function
def invoke_pipeline(messages, top_n=5, output_format="console"):
    results = []
    for role, message in messages:
        if role.lower() == 'user':
            result = user_proxy.handle_query(
                user_query=message,
                top_n=top_n,
                output_format=output_format
            )
            results.append((role, result))
        else:
            results.append((role, f"No handler for role: {role}"))
    return results

# 5️⃣ Example usage
output = invoke_pipeline([('user', 'Top 5 stores within 1 km of 25.198, 55.279')])

# 6️⃣ Inspect output
for role, content in output:
    print(f"{role.upper()} RESPONSE:")
    print(content)

##Agent orchestration and pipeline execution

In [ ]:
analytics_agent = AnalyticsAgent(stores)
report_agent = ReportAgent(analytics_agent)
output_agent = OutputAgent(report_agent)
critic_agent = CriticAgent(analytics_agent)

user_proxy = UserProxyAgent(
    query_agent=query_agent,
    analytics_agent=analytics_agent,
    report_agent=report_agent,
    output_agent=output_agent,
    critic_agent=critic_agent
)

# Simple pipeline invocation function
def invoke_pipeline(messages, top_n=5, output_format="dict"):
    """
    messages: list of tuples [('role', 'message')]
    Returns conversation log and report
    """
    results = []
    for role, message in messages:
        if role.lower() == "user":
            result = user_proxy.handle_query(
                user_query=message,
                top_n=top_n,
                output_format=output_format
            )
            results.append((role, result))
        else:
            results.append((role, f"No handler for role: {role}"))
    return results

# Invoke the pipeline with a user query
output = invoke_pipeline([
    ('user', 'Top 5 stores within 1 km of 25.198, 55.279')
], top_n=5, output_format="dict")

# Display the conversation
for role, content in output:
    print(f"\n{role.upper()} RESPONSE:")
    for msg in content["conversation"]:
        print(f"{msg['agent']}: {msg['message']}")

    print("\nReport Summary:")
    print(content["report_summary"])

    print("\nTop Stores Table:")
    for store in content["top_stores"]:
        print(store)

##References

- Analytics Vidhya for LangGraph concepts
- ChatGPT for code